In [1]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime

# Load all JSON files
output_dir = Path('.')
results = {}

for json_file in sorted(output_dir.glob('*.json')):
    with open(json_file) as f:
        data = json.load(f)
        
        # Create unique identifier: model name + timestamp from data
        model_name = data['model']
        timestamp = data.get('timestamp', '')
        
        # Parse timestamp to create readable suffix
        if timestamp:
            try:
                dt = datetime.fromisoformat(timestamp)
                time_suffix = dt.strftime('%m/%d %H:%M')
                experiment_id = f"{model_name} ({time_suffix})"
            except:
                # Fallback to filename if timestamp parsing fails
                experiment_id = json_file.stem.replace('manipulation_', '')
        else:
            experiment_id = json_file.stem.replace('manipulation_', '')
        
        results[experiment_id] = data

print(f"Loaded {len(results)} experiments:")
for exp_id in results.keys():
    print(f"  - {exp_id}")

Loaded 4 experiments:
  - @azure-openai-foundry/gpt-4.1 (02/09 00:45)
  - @azure-openai-foundry/gpt-4.1 (02/09 00:50)
  - @azure-openai-foundry/gpt-4.1 (02/09 01:05)
  - llama3.1:8b (02/09 01:08)


In [2]:
# Create comparison tables
datasets = list(next(iter(results.values()))['datasets'].keys())

# Table 1: Macro F1 scores across datasets
f1_data = []
for model_name, model_data in results.items():
    row = {'Model': model_name}
    for dataset in datasets:
        if dataset in model_data['datasets']:
            f1 = model_data['datasets'][dataset]['reports']['original']['macro avg']['f1-score']
            row[dataset] = round(f1, 3)
    f1_data.append(row)

df_f1 = pd.DataFrame(f1_data).set_index('Model')
print("\n=== Macro F1 Scores by Model and Dataset ===")
print(df_f1.to_string())

# Table 2: Accuracy across datasets
acc_data = []
for model_name, model_data in results.items():
    row = {'Model': model_name}
    for dataset in datasets:
        if dataset in model_data['datasets']:
            acc = model_data['datasets'][dataset]['reports']['original']['accuracy']
            row[dataset] = round(acc, 3)
    acc_data.append(row)

df_acc = pd.DataFrame(acc_data).set_index('Model')
print("\n=== Accuracy by Model and Dataset ===")
print(df_acc.to_string())

# Summary statistics
print("\n=== Average Performance Across All Datasets ===")
summary = pd.DataFrame({
    'Model': list(results.keys()),
    'Avg Macro F1': [df_f1.loc[model].mean() for model in results.keys()],
    'Avg Accuracy': [df_acc.loc[model].mean() for model in results.keys()]
}).set_index('Model')
print(summary.round(3).to_string())

# Optional: Display styled dataframes if running in Jupyter
display(df_f1.style.highlight_max(axis=0, color='lightgreen').format("{:.3f}"))
display(df_acc.style.highlight_max(axis=0, color='lightgreen').format("{:.3f}"))


=== Macro F1 Scores by Model and Dataset ===
                                             ABSTRCT  ACQUA    AEC    AFS  ARGUMINSCI  FINARG    IAM     PE  SCIARK  USELEC
Model                                                                                                                      
@azure-openai-foundry/gpt-4.1 (02/09 00:45)    1.000  0.422  0.583  0.800       0.800   0.524  0.697  0.495   0.697    0.80
@azure-openai-foundry/gpt-4.1 (02/09 00:50)    1.000  0.422  0.583  0.800       0.899   0.524  0.600  0.451   0.583    0.80
@azure-openai-foundry/gpt-4.1 (02/09 01:05)    1.000  0.600  0.495  0.800       0.899   0.524  0.697  0.400   0.697    0.80
llama3.1:8b (02/09 01:08)                      0.899  0.670  0.524  0.524       0.800   0.286  0.670  0.333   0.451    0.67

=== Accuracy by Model and Dataset ===
                                             ABSTRCT  ACQUA  AEC  AFS  ARGUMINSCI  FINARG  IAM   PE  SCIARK  USELEC
Model                                                  

,ABSTRCT,ACQUA,AEC,AFS,ARGUMINSCI,FINARG,IAM,PE,SCIARK,USELEC
Model,,,,,,,,,,
@azure-openai-foundry/gpt-4.1 (02/09 00:45),1.000,0.422,0.583,0.800,0.800,0.524,0.697,0.495,0.697,0.800
@azure-openai-foundry/gpt-4.1 (02/09 00:50),1.000,0.422,0.583,0.800,0.899,0.524,0.600,0.451,0.583,0.800
@azure-openai-foundry/gpt-4.1 (02/09 01:05),1.000,0.600,0.495,0.800,0.899,0.524,0.697,0.400,0.697,0.800
llama3.1:8b (02/09 01:08),0.899,0.670,0.524,0.524,0.800,0.286,0.670,0.333,0.451,0.670


,ABSTRCT,ACQUA,AEC,AFS,ARGUMINSCI,FINARG,IAM,PE,SCIARK,USELEC
Model,,,,,,,,,,
@azure-openai-foundry/gpt-4.1 (02/09 00:45),1.000,0.600,0.600,0.800,0.800,0.600,0.700,0.500,0.700,0.800
@azure-openai-foundry/gpt-4.1 (02/09 00:50),1.000,0.600,0.600,0.800,0.900,0.600,0.600,0.500,0.600,0.800
@azure-openai-foundry/gpt-4.1 (02/09 01:05),1.000,0.600,0.500,0.800,0.900,0.600,0.700,0.400,0.700,0.800
llama3.1:8b (02/09 01:08),0.900,0.700,0.600,0.600,0.800,0.400,0.700,0.500,0.500,0.700


In [3]:
# Robustness Analysis: Macro F1 Deltas (Manipulation - Original)
# Negative delta = performance drop under manipulation (less robust)
# Positive delta = performance improvement (more robust)
# Close to 0 = most robust

manipulation_types = ['feger', 'shuffle']

for manip_type in manipulation_types:
    print(f"\n{'='*60}")
    print(f"=== Robustness to '{manip_type.upper()}' Manipulation ===")
    print(f"{'='*60}")
    
    delta_data = []
    for model_name, model_data in results.items():
        row = {'Model': model_name}
        for dataset in datasets:
            if dataset in model_data['datasets']:
                reports = model_data['datasets'][dataset]['reports']
                if manip_type in reports:
                    original_f1 = reports['original']['macro avg']['f1-score']
                    manip_f1 = reports[manip_type]['macro avg']['f1-score']
                    delta = manip_f1 - original_f1  # Negative = performance drop
                    row[dataset] = round(delta, 3)
        delta_data.append(row)
    
    df_delta = pd.DataFrame(delta_data).set_index('Model')
    print(f"\nMacro F1 Delta ({manip_type.capitalize()} - Original):")
    print(df_delta.to_string())
    
    # Summary: Average delta across all datasets
    print(f"\n=== Average Robustness (Closer to 0 = More Robust) ===")
    avg_delta = pd.DataFrame({
        'Model': list(results.keys()),
        'Avg F1 Delta': [df_delta.loc[model].mean() for model in results.keys()]
    }).set_index('Model').sort_values('Avg F1 Delta', ascending=False)
    print(avg_delta.round(3).to_string())
    
    # Display styled dataframe (closest to 0 = most robust)
    # Highlight values closest to 0 by creating absolute value styling
    styled = df_delta.style.background_gradient(cmap='RdYlGn', axis=None, vmin=-0.5, vmax=0.5)
    display(styled.format("{:.3f}"))


=== Robustness to 'FEGER' Manipulation ===

Macro F1 Delta (Feger - Original):
                                             ABSTRCT  ACQUA    AEC    AFS  ARGUMINSCI  FINARG    IAM     PE  SCIARK  USELEC
Model                                                                                                                      
@azure-openai-foundry/gpt-4.1 (02/09 00:45)   -0.101  0.073 -0.250 -0.276      -0.130  -0.190 -0.097  0.029  -0.097  -0.130
@azure-openai-foundry/gpt-4.1 (02/09 00:50)   -0.208  0.073 -0.250 -0.276      -0.107  -0.190  0.000  0.220  -0.088  -0.130
@azure-openai-foundry/gpt-4.1 (02/09 01:05)    0.000 -0.105 -0.209 -0.276      -0.375  -0.190 -0.097 -0.067  -0.246  -0.130
llama3.1:8b (02/09 01:08)                     -0.107  0.027  0.076 -0.073      -0.103   0.007  0.121  0.267   0.349   0.027

=== Average Robustness (Closer to 0 = More Robust) ===
                                             Avg F1 Delta
Model                                                    
llam

,ABSTRCT,ACQUA,AEC,AFS,ARGUMINSCI,FINARG,IAM,PE,SCIARK,USELEC
Model,,,,,,,,,,
@azure-openai-foundry/gpt-4.1 (02/09 00:45),-0.101,0.073,-0.250,-0.276,-0.130,-0.190,-0.097,0.029,-0.097,-0.130
@azure-openai-foundry/gpt-4.1 (02/09 00:50),-0.208,0.073,-0.250,-0.276,-0.107,-0.190,0.000,0.220,-0.088,-0.130
@azure-openai-foundry/gpt-4.1 (02/09 01:05),0.000,-0.105,-0.209,-0.276,-0.375,-0.190,-0.097,-0.067,-0.246,-0.130
llama3.1:8b (02/09 01:08),-0.107,0.027,0.076,-0.073,-0.103,0.007,0.121,0.267,0.349,0.027



=== Robustness to 'SHUFFLE' Manipulation ===

Macro F1 Delta (Shuffle - Original):
                                             ABSTRCT  ACQUA    AEC    AFS  ARGUMINSCI  FINARG    IAM     PE  SCIARK  USELEC
Model                                                                                                                      
@azure-openai-foundry/gpt-4.1 (02/09 00:45)   -0.330  0.275 -0.250 -0.467      -0.276  -0.190 -0.027 -0.162  -0.173  -0.276
@azure-openai-foundry/gpt-4.1 (02/09 00:50)   -0.208  0.178 -0.250 -0.467      -0.566  -0.190 -0.017 -0.165   0.087  -0.130
@azure-openai-foundry/gpt-4.1 (02/09 01:05)   -0.101  0.097 -0.304 -0.467      -0.566  -0.190 -0.114  0.124  -0.364  -0.276
llama3.1:8b (02/09 01:08)                     -0.202 -0.087  0.060  0.147      -0.217  -0.055 -0.070  0.337   0.220   0.130

=== Average Robustness (Closer to 0 = More Robust) ===
                                             Avg F1 Delta
Model                                                    


,ABSTRCT,ACQUA,AEC,AFS,ARGUMINSCI,FINARG,IAM,PE,SCIARK,USELEC
Model,,,,,,,,,,
@azure-openai-foundry/gpt-4.1 (02/09 00:45),-0.330,0.275,-0.250,-0.467,-0.276,-0.190,-0.027,-0.162,-0.173,-0.276
@azure-openai-foundry/gpt-4.1 (02/09 00:50),-0.208,0.178,-0.250,-0.467,-0.566,-0.190,-0.017,-0.165,0.087,-0.130
@azure-openai-foundry/gpt-4.1 (02/09 01:05),-0.101,0.097,-0.304,-0.467,-0.566,-0.190,-0.114,0.124,-0.364,-0.276
llama3.1:8b (02/09 01:08),-0.202,-0.087,0.060,0.147,-0.217,-0.055,-0.070,0.337,0.220,0.130
